# POS delay — raffinamento dei confini delle effect window

Questo notebook valuta **solo** la qualità delle effect window POS delay già rilevate dalla pipeline principale.

La procedura è separata dalla valutazione event-level della sensitivity analysis:

1. riusa il modello LSTM POS e la configurazione fissa del detector;
2. esegue inference, costruzione delle profile window e detection come nel notebook di sensitivity;
3. considera soltanto le finestre rilevate abbinate a una effect window ground truth, esclusivamente per la valutazione dei confini;
4. raffina ogni finestra tramite segmentazione locale dello score `pos_cos`;
5. confronta finestra raw e finestra raffinata tramite IoU, offset di inizio e fine, ed errore sulla durata.

Il refinement non utilizza ground truth, tipo del ritardo o source day. Tali informazioni intervengono soltanto nel matching necessario a misurare la qualità dei confini.

In [1]:
# =========================================================
# PATH
# =========================================================

import sys
from pathlib import Path

start_dir = Path.cwd().resolve()

for candidate in [start_dir, *start_dir.parents]:
    if (candidate / "project_paths.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Root del progetto non trovata. "
        "Avvia Jupyter da una cartella interna alla repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/Users/ciok4/jupyter file/tesi')

## Import e dipendenze

Gli artifact e i path sono centralizzati in `project_paths.py`. Le funzioni comuni per inference, profile window, detector e cache sono importate da `POS_delay_utils.py`.

In [2]:
# =========================================================
# IMPORTS
# =========================================================

import pickle
import time

import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display
from tqdm.auto import tqdm

from project_paths import (
    POS_DELAY_RESULTS_DIR,
    POS_DELAY_SENSITIVITY_DIR,
    POS_MODEL_DIR,
    ensure_artifact_directories,
)

from lstm_utils import interval_iou

from POS_delay_utils import (
    POS_DELAY_DETECTOR_CONFIG,
    add_detection_offsets,
    compute_results_for_dataset,
    list_pos_delay_sensitivity_datasets,
    make_pos_delay_cache_path,
    run_detector_config_on_results,
)

pd.set_option("display.max_columns", None)

## Configurazione e cache

La configurazione del detector coincide con quella fissata nella sensitivity analysis POS.  
La cache conserva solo i risultati dell'inference LSTM; il refinement viene rieseguito quando `FORCE_RECOMPUTE_REFINEMENT = True` oppure quando manca il CSV event-level.

In [3]:
# =========================================================
# CONFIG
# =========================================================

ensure_artifact_directories()

BASE_SENSITIVITY_PATH = POS_DELAY_SENSITIVITY_DIR
MODEL_DIR = POS_MODEL_DIR

OUTPUT_DIR = POS_DELAY_RESULTS_DIR / "boundary_refinement"
INFERENCE_CACHE_DIR = OUTPUT_DIR / "_inference_cache"
TABLES_DIR = OUTPUT_DIR / "tables"

for path in [OUTPUT_DIR, INFERENCE_CACHE_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# L'analisi dei confini resta riferita al caso principale con un solo source day.
SOURCE_DURATION_FILTER = [1]

DELAY_TYPES_FILTER = [
    "mild_delay",
    "moderate_delay",
    "strong_delay",
    "batch_backlog",
    "settlement_freeze",
]

# Finestra temporale del modello LSTM POS selezionato.
WINDOW_SIZE = 7

# Configurazione fissa del detector POS delay.
DETECTOR_CONFIG = POS_DELAY_DETECTOR_CONFIG.copy()
MATCH_IOU_THRESHOLD = float(DETECTOR_CONFIG["iou_threshold"])

# -------------------------
# Boundary refinement
# -------------------------

# Numero di profile window aggiunte a ciascun lato della detection raw.
REFINEMENT_MARGIN_PROFILE_WINDOWS = 3

# Vincoli minimi della segmentazione a tre regimi.
MIN_REFINED_PROFILE_WINDOWS = 1
MIN_SIDE_PROFILE_WINDOWS = 2
MIN_POS_COS_EFFECT_STD_FRACTION = 0.25

# Con False, la finestra raffinata può solo restringere quella rilevata.
ALLOW_BOUNDARY_EXPANSION = True

# -------------------------
# Cache / debug
# -------------------------

# Ricalcola l'inference LSTM anche se esistono cache per dataset.
FORCE_RECOMPUTE_INFERENCE = False

# Ignora il CSV dei risultati e riesegue detection, matching e refinement.
FORCE_RECOMPUTE_REFINEMENT = False

# Utile per controlli rapidi. Lasciare None per l'analisi completa.
DEBUG_MAX_DATASETS = None

RAW_MATCHED_RESULTS_PATH = (
    TABLES_DIR / "pos_delay_boundary_refinement_matched_events.csv"
)
OVERALL_SUMMARY_PATH = (
    TABLES_DIR / "pos_delay_boundary_refinement_overall_summary.csv"
)
STATUS_SUMMARY_PATH = (
    TABLES_DIR / "pos_delay_boundary_refinement_status_summary.csv"
)
THESIS_SUMMARY_PATH = (
    TABLES_DIR / "pos_delay_boundary_refinement_thesis_summary.csv"
)

CONFIG = {
    "source_duration_filter": SOURCE_DURATION_FILTER,
    "delay_types_filter": DELAY_TYPES_FILTER,
    "window_size": WINDOW_SIZE,
    "detector_config": DETECTOR_CONFIG,
    "match_iou_threshold": MATCH_IOU_THRESHOLD,
    "refinement_margin_profile_windows": REFINEMENT_MARGIN_PROFILE_WINDOWS,
    "min_refined_profile_windows": MIN_REFINED_PROFILE_WINDOWS,
    "min_side_profile_windows": MIN_SIDE_PROFILE_WINDOWS,
    "min_pos_cos_effect_std_fraction": MIN_POS_COS_EFFECT_STD_FRACTION,
    "allow_boundary_expansion": ALLOW_BOUNDARY_EXPANSION,
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_SENSITIVITY_PATH:", BASE_SENSITIVITY_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
display(pd.Series(CONFIG))

PROJECT_ROOT: C:\Users\ciok4\jupyter file\tesi
BASE_SENSITIVITY_PATH: C:\Users\ciok4\jupyter file\tesi\artifacts\data\datasets\pos_delay\sensitivity
MODEL_DIR: C:\Users\ciok4\jupyter file\tesi\artifacts\models\pos
OUTPUT_DIR: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\boundary_refinement


source_duration_filter                                                             [1]
delay_types_filter                   [mild_delay, moderate_delay, strong_delay, bat...
window_size                                                                          7
detector_config                      {'score_col': 'pos_cos', 'profile_window_size'...
match_iou_threshold                                                                0.2
refinement_margin_profile_windows                                                    3
min_refined_profile_windows                                                          1
min_side_profile_windows                                                             2
min_pos_cos_effect_std_fraction                                                   0.25
allow_boundary_expansion                                                          True
dtype: object

## Modello e dataset di sensitivity

Vengono caricati il modello POS selezionato e gli artifact di preprocessing. I dataset seguono la stessa struttura usata nel notebook di sensitivity.

In [4]:
# =========================================================
# LOAD MODEL + PREPROCESSING ARTIFACTS
# =========================================================

model = tf.keras.models.load_model(
    MODEL_DIR / "lstm_pos.keras"
)

with open(MODEL_DIR / "feature_scalers.pkl", "rb") as f:
    feature_scalers = pickle.load(f)

with open(MODEL_DIR / "mappings.pkl", "rb") as f:
    mappings = pickle.load(f)

with open(MODEL_DIR / "features.pkl", "rb") as f:
    features = pickle.load(f)

In [5]:
# =========================================================
# LIST SENSITIVITY DATASETS
# =========================================================

datasets_df = list_pos_delay_sensitivity_datasets(
    BASE_SENSITIVITY_PATH,
    source_duration_filter=SOURCE_DURATION_FILTER,
    delay_types_filter=DELAY_TYPES_FILTER,
)

if datasets_df.empty:
    raise FileNotFoundError(
        "Nessun dataset POS delay trovato. Controlla BASE_SENSITIVITY_PATH "
        "e i filtri SOURCE_DURATION_FILTER / DELAY_TYPES_FILTER."
    )

datasets_df = datasets_df.sort_values(
    ["delay_type", "source_duration", "seed"]
).reset_index(drop=True)

if DEBUG_MAX_DATASETS is not None:
    datasets_df = datasets_df.head(DEBUG_MAX_DATASETS).copy()

print("Numero dataset:", len(datasets_df))
display(datasets_df.head())

experiment_coverage = (
    datasets_df
    .groupby(["source_duration", "delay_type"])
    .size()
    .rename("n_datasets")
    .reset_index()
)

display(experiment_coverage)

Numero dataset: 25


,path,delay_type,source_duration,seed
0,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,42
1,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,43
2,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,44
3,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,45
4,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,46


,source_duration,delay_type,n_datasets
0,1,batch_backlog,5
1,1,mild_delay,5
2,1,moderate_delay,5
3,1,settlement_freeze,5
4,1,strong_delay,5


## Inference, detection e matching

L'inference viene riutilizzata da cache per ciascun dataset. Il detector è quello centralizzato e già usato nella sensitivity analysis; il matching identifica soltanto le detection confrontabili con una effect window reale.

In [6]:
# =========================================================
# INFERENCE CACHE + MATCHED WINDOWS
# =========================================================

def compute_or_load_results(dataset_row):
    """
    Carica da cache oppure calcola train, validation e test results per un
    dataset di sensitivity.
    """

    cache_path = make_pos_delay_cache_path(
        dataset_row,
        INFERENCE_CACHE_DIR,
    )

    if cache_path.exists() and not FORCE_RECOMPUTE_INFERENCE:
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    dataset_cache = compute_results_for_dataset(
        csv_path=dataset_row["path"],
        model=model,
        feature_scalers=feature_scalers,
        mappings=mappings,
        features=features,
        window_size=WINDOW_SIZE,
    )

    with open(cache_path, "wb") as f:
        pickle.dump(dataset_cache, f)

    return dataset_cache


def run_detector_and_get_matches(dataset_cache):
    """
    Applica il detector POS fisso e restituisce le sole ground truth abbinate
    alle detection, con i confini raw della finestra rilevata.
    """

    detector_output = run_detector_config_on_results(
        val_results=dataset_cache["val_results"],
        test_results=dataset_cache["test_results"],
        profile_window_size=int(DETECTOR_CONFIG["profile_window_size"]),
        score_col=DETECTOR_CONFIG["score_col"],
        z_threshold=float(DETECTOR_CONFIG["z_threshold"]),
        min_consecutive=int(DETECTOR_CONFIG["min_consecutive"]),
        gap_tolerance=int(DETECTOR_CONFIG["gap_tolerance"]),
        iou_threshold=MATCH_IOU_THRESHOLD,
    )

    if detector_output is None:
        return None, pd.DataFrame()

    gt_eval = add_detection_offsets(
        detector_output["gt_eval"],
        detector_output["det_eval"],
    )

    matched = gt_eval.loc[
        gt_eval["matched"].astype(int) == 1
    ].copy()

    return detector_output, matched

## Raffinamento della effect window

Il refinement applica una segmentazione locale a tre regimi alla sequenza degli score `pos_cos` delle profile window:

\[
\text{profilo normale prima}
\quad | \quad
\text{profilo più dissimile}
\quad | \quad
\text{profilo normale dopo}.
\]

Tra le segmentazioni possibili viene scelta quella con minore somma degli errori quadratici, imponendo che il segmento centrale abbia uno score medio superiore al contesto laterale. La ground truth non interviene nella scelta dei nuovi confini.

In [7]:
# =========================================================
# EFFECT WINDOW REFINEMENT
# =========================================================

def _segment_sse(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return np.inf

    mean_value = np.nanmean(values)
    return np.nansum((values - mean_value) ** 2)


def _mean_safe(values):
    values = np.asarray(values, dtype=float)
    return np.nanmean(values) if len(values) > 0 else np.nan


def _count_business_days(store_test_results, start, end):
    dates = pd.to_datetime(store_test_results["date"])

    return int(
        (
            (store_test_results["holiday"].astype(int) == 0)
            & (dates >= pd.to_datetime(start))
            & (dates <= pd.to_datetime(end))
        ).sum()
    )


def refine_effect_window_by_local_pos_cos_segmentation(
    store_test_profiles,
    store_test_results,
    raw_start,
    raw_end,
    score_col="pos_cos",
    refinement_margin_profile_windows=REFINEMENT_MARGIN_PROFILE_WINDOWS,
    min_refined_profile_windows=MIN_REFINED_PROFILE_WINDOWS,
    min_side_profile_windows=MIN_SIDE_PROFILE_WINDOWS,
    min_effect_std_fraction=MIN_POS_COS_EFFECT_STD_FRACTION,
    allow_boundary_expansion=ALLOW_BOUNDARY_EXPANSION,
):
    """
    Raffina una detection raw tramite segmentazione locale a tre regimi degli
    score di profilo. La ground truth non è usata in questa funzione.
    """

    raw_start = pd.to_datetime(raw_start)
    raw_end = pd.to_datetime(raw_end)

    profiles = store_test_profiles.copy()

    if profiles.empty:
        return raw_start, raw_end, "fallback_no_profiles", 0

    for column in ["window_start", "window_end", "center_date"]:
        profiles[column] = pd.to_datetime(profiles[column])

    profiles = profiles.sort_values("center_date").reset_index(drop=True)

    # Profile window che costituiscono o intersecano la detection raw.
    raw_overlap = (
        (profiles["window_end"] >= raw_start)
        & (profiles["window_start"] <= raw_end)
    )

    raw_positions = np.flatnonzero(raw_overlap.to_numpy())

    if len(raw_positions) == 0:
        return raw_start, raw_end, "fallback_no_raw_profile", 0

    local_left = max(
        0,
        raw_positions.min() - refinement_margin_profile_windows,
    )
    local_right = min(
        len(profiles) - 1,
        raw_positions.max() + refinement_margin_profile_windows,
    )

    local = (
        profiles
        .iloc[local_left:local_right + 1]
        .copy()
        .reset_index(drop=True)
    )

    minimum_length = (
        2 * min_side_profile_windows + min_refined_profile_windows
    )

    if len(local) < minimum_length:
        return raw_start, raw_end, "fallback_too_short", 0

    scores = local[score_col].astype(float).to_numpy()
    local_std = np.nanstd(scores)
    minimum_effect = min_effect_std_fraction * local_std

    candidates = []
    relaxed_candidates = []
    n_profiles = len(local)

    for start_idx in range(
        min_side_profile_windows,
        n_profiles - min_side_profile_windows - min_refined_profile_windows + 1,
    ):
        for end_idx in range(
            start_idx + min_refined_profile_windows - 1,
            n_profiles - min_side_profile_windows,
        ):
            left_scores = scores[:start_idx]
            middle_scores = scores[start_idx:end_idx + 1]
            right_scores = scores[end_idx + 1:]

            candidate_profiles = local.iloc[start_idx:end_idx + 1]

            if allow_boundary_expansion:
                overlaps_raw = (
                    candidate_profiles["window_end"].max() >= raw_start
                    and candidate_profiles["window_start"].min() <= raw_end
                )

                if not overlaps_raw:
                    continue
            else:
                is_contained_in_raw = (
                    (candidate_profiles["window_start"] >= raw_start).all()
                    and (candidate_profiles["window_end"] <= raw_end).all()
                )

                if not is_contained_in_raw:
                    continue

            outside_scores = np.concatenate([left_scores, right_scores])

            middle_mean = _mean_safe(middle_scores)
            outside_mean = _mean_safe(outside_scores)
            effect = middle_mean - outside_mean

            # pos_cos è una dissimilarità: il segmento anomalo deve avere
            # score medio superiore rispetto al contesto laterale.
            effect_sign_ok = effect > 0
            effect_size_ok = effect >= minimum_effect

            objective = (
                _segment_sse(left_scores)
                + _segment_sse(middle_scores)
                + _segment_sse(right_scores)
            )

            candidate = {
                "start_idx": start_idx,
                "end_idx": end_idx,
                "objective": objective,
                "effect": effect,
            }

            if effect_sign_ok:
                relaxed_candidates.append(candidate)

            if effect_sign_ok and effect_size_ok:
                candidates.append(candidate)

    if candidates:
        best = min(candidates, key=lambda candidate: candidate["objective"])
        status = "refined_pos_cos"
    elif relaxed_candidates:
        best = min(
            relaxed_candidates,
            key=lambda candidate: candidate["objective"],
        )
        status = "refined_pos_cos_relaxed_effect"
    else:
        return raw_start, raw_end, "fallback_no_valid_segment", 0

    refined_profiles = local.iloc[
        best["start_idx"]:best["end_idx"] + 1
    ]

    refined_start = pd.to_datetime(refined_profiles["window_start"].min())
    refined_end = pd.to_datetime(refined_profiles["window_end"].max())

    if not allow_boundary_expansion:
        refined_start = max(refined_start, raw_start)
        refined_end = min(refined_end, raw_end)

    refined_n_days = _count_business_days(
        store_test_results,
        refined_start,
        refined_end,
    )

    return (
        refined_start,
        refined_end,
        status,
        refined_n_days,
    )


def add_boundary_metrics(row):
    raw_start = pd.to_datetime(row["raw_start"])
    raw_end = pd.to_datetime(row["raw_end"])
    refined_start = pd.to_datetime(row["refined_start"])
    refined_end = pd.to_datetime(row["refined_end"])
    gt_start = pd.to_datetime(row["gt_start"])
    gt_end = pd.to_datetime(row["gt_end"])

    return {
        "raw_offset_start": (raw_start - gt_start).days,
        "raw_offset_end": (raw_end - gt_end).days,
        "refined_offset_start": (refined_start - gt_start).days,
        "refined_offset_end": (refined_end - gt_end).days,
        "raw_iou": interval_iou(raw_start, raw_end, gt_start, gt_end),
        "refined_iou": interval_iou(
            refined_start,
            refined_end,
            gt_start,
            gt_end,
        ),
    }

## Valutazione delle finestre abbinate

Il refinement viene applicato solo alle detection già abbinate a una effect window ground truth. La ground truth serve esclusivamente per confrontare i confini raw e raffinati dopo l'applicazione della segmentazione.

In [8]:
# =========================================================
# MAIN LOOP: MATCHING + BOUNDARY REFINEMENT
# =========================================================

if RAW_MATCHED_RESULTS_PATH.exists() and not FORCE_RECOMPUTE_REFINEMENT:
    event_results = pd.read_csv(RAW_MATCHED_RESULTS_PATH)

    for column in [
        "gt_start",
        "gt_end",
        "raw_start",
        "raw_end",
        "refined_start",
        "refined_end",
    ]:
        if column in event_results.columns:
            event_results[column] = pd.to_datetime(event_results[column])

    print("Risultati del refinement già disponibili:", RAW_MATCHED_RESULTS_PATH)

else:
    event_rows = []
    start_time = time.time()

    for _, dataset_row in tqdm(
        datasets_df.iterrows(),
        total=len(datasets_df),
        desc="Detection and boundary refinement",
    ):
        dataset_cache = compute_or_load_results(dataset_row)

        detector_output, matched_events = run_detector_and_get_matches(
            dataset_cache
        )

        if detector_output is None or matched_events.empty:
            continue

        test_results = dataset_cache["test_results"]

        for _, matched_row in matched_events.iterrows():
            store_id = matched_row["store_id"]

            store_profiles = (
                detector_output["test_profiles_cmp"]
                .loc[
                    lambda frame: frame["store_id"] == store_id
                ]
                .sort_values("center_date")
                .reset_index(drop=True)
            )

            store_test_results = (
                test_results
                .loc[
                    lambda frame: frame["store_id"] == store_id
                ]
                .sort_values("date")
                .reset_index(drop=True)
            )

            refined_start, refined_end, refinement_status, refined_n_days = (
                refine_effect_window_by_local_pos_cos_segmentation(
                    store_test_profiles=store_profiles,
                    store_test_results=store_test_results,
                    raw_start=matched_row["detected_start"],
                    raw_end=matched_row["detected_end"],
                    score_col=DETECTOR_CONFIG["score_col"],
                )
            )

            event_row = {
                "dataset_path": str(dataset_row["path"]),
                "source_duration": int(dataset_row["source_duration"]),
                "seed": int(dataset_row["seed"]),
                "store_id": store_id,
                "gt_event_id": matched_row["gt_event_id"],
                "detected_id": int(matched_row["matched_detected_id"]),
                "gt_start": pd.to_datetime(matched_row["gt_start"]),
                "gt_end": pd.to_datetime(matched_row["gt_end"]),
                "raw_start": pd.to_datetime(matched_row["detected_start"]),
                "raw_end": pd.to_datetime(matched_row["detected_end"]),
                "refined_start": refined_start,
                "refined_end": refined_end,
                "refinement_status": refinement_status,
                "refined_n_business_days": refined_n_days,
            }

            event_row.update(add_boundary_metrics(event_row))
            event_rows.append(event_row)

    event_results = pd.DataFrame(event_rows)

    if not event_results.empty:
        event_results = event_results.sort_values(
            [
                "source_duration",
                "seed",
                "store_id",
                "detected_id",
            ]
        ).reset_index(drop=True)

        event_results.to_csv(RAW_MATCHED_RESULTS_PATH, index=False)

    elapsed = time.time() - start_time
    print(f"Completato in {elapsed / 60:.2f} minuti.")
    print("Eventi matched analizzati:", len(event_results))

Risultati del refinement già disponibili: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\boundary_refinement\tables\pos_delay_boundary_refinement_matched_events.csv


## Salvataggio e sintesi

Le tabelle riportano soltanto la qualità dei confini delle finestre raw e raffinate. Non vengono prodotti stime di source day, classificazioni del tipo di ritardo o risultati Monte Carlo.

In [9]:
# =========================================================
# SUMMARY TABLES
# =========================================================

def summarize_boundary_refinement(df):
    if df.empty:
        return pd.DataFrame()

    temp = df.copy()

    temp["raw_abs_offset_start"] = temp["raw_offset_start"].abs()
    temp["raw_abs_offset_end"] = temp["raw_offset_end"].abs()
    temp["refined_abs_offset_start"] = (
        temp["refined_offset_start"].abs()
    )
    temp["refined_abs_offset_end"] = (
        temp["refined_offset_end"].abs()
    )
    temp["iou_delta"] = temp["refined_iou"] - temp["raw_iou"]

    return pd.DataFrame([{
        "n_events": len(temp),
        "n_refined": int(
            temp["refinement_status"].str.startswith("refined").sum()
        ),
        "n_fallback": int(
            temp["refinement_status"].str.startswith("fallback").sum()
        ),
        "raw_iou_mean": temp["raw_iou"].mean(),
        "refined_iou_mean": temp["refined_iou"].mean(),
        "iou_delta_mean": temp["iou_delta"].mean(),
        "raw_abs_offset_start_mean": temp["raw_abs_offset_start"].mean(),
        "refined_abs_offset_start_mean": (
            temp["refined_abs_offset_start"].mean()
        ),
        "raw_abs_offset_end_mean": temp["raw_abs_offset_end"].mean(),
        "refined_abs_offset_end_mean": (
            temp["refined_abs_offset_end"].mean()
        ),
    }])


if event_results.empty:
    overall_summary = pd.DataFrame()
    status_summary = pd.DataFrame()
    print("Nessun evento matched disponibile.")

else:
    overall_summary = summarize_boundary_refinement(event_results)

    status_summary = (
        event_results
        .groupby("refinement_status", as_index=False)
        .size()
        .rename(columns={"size": "n_events"})
        .sort_values("n_events", ascending=False)
        .reset_index(drop=True)
    )

    overall_summary.to_csv(OVERALL_SUMMARY_PATH, index=False)
    status_summary.to_csv(STATUS_SUMMARY_PATH, index=False)
    print("Salvato:", RAW_MATCHED_RESULTS_PATH)
    print("Salvato:", OVERALL_SUMMARY_PATH)
    print("Salvato:", STATUS_SUMMARY_PATH)

Salvato: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\boundary_refinement\tables\pos_delay_boundary_refinement_matched_events.csv
Salvato: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\boundary_refinement\tables\pos_delay_boundary_refinement_overall_summary.csv
Salvato: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\boundary_refinement\tables\pos_delay_boundary_refinement_status_summary.csv


## Tabella riassuntiva

La tabella riassume il confronto diretto tra i confini raw e quelli raffinati sugli stessi eventi matched.

In [10]:
# =========================================================
# THESIS SUMMARY
# =========================================================

if THESIS_SUMMARY_PATH.exists():
    thesis_boundary_summary = pd.read_csv(THESIS_SUMMARY_PATH)

    print("Tabella già disponibile:", THESIS_SUMMARY_PATH)
    display(thesis_boundary_summary.round(3))

elif event_results.empty:
    print("event_results è vuoto.")

else:
    boundary_eval = event_results.copy()

    for column in [
        "gt_start",
        "gt_end",
        "raw_start",
        "raw_end",
        "refined_start",
        "refined_end",
    ]:
        boundary_eval[column] = pd.to_datetime(boundary_eval[column])

    boundary_eval["gt_duration_days"] = (
        boundary_eval["gt_end"] - boundary_eval["gt_start"]
    ).dt.days + 1

    boundary_eval["raw_duration_days"] = (
        boundary_eval["raw_end"] - boundary_eval["raw_start"]
    ).dt.days + 1

    boundary_eval["refined_duration_days"] = (
        boundary_eval["refined_end"] - boundary_eval["refined_start"]
    ).dt.days + 1

    raw_abs_duration_error_mean = (
        boundary_eval["raw_duration_days"]
        .sub(boundary_eval["gt_duration_days"])
        .abs()
        .mean()
    )

    refined_abs_duration_error_mean = (
        boundary_eval["refined_duration_days"]
        .sub(boundary_eval["gt_duration_days"])
        .abs()
        .mean()
    )

    summary_row = overall_summary.iloc[0]

    thesis_boundary_summary = pd.DataFrame(
        {
            "Metrica": [
                "IoU media",
                "Errore assoluto inizio (giorni)",
                "Errore assoluto fine (giorni)",
                "Errore assoluto durata (giorni)",
            ],
            "Finestra raw": [
                summary_row["raw_iou_mean"],
                summary_row["raw_abs_offset_start_mean"],
                summary_row["raw_abs_offset_end_mean"],
                raw_abs_duration_error_mean,
            ],
            "Finestra raffinata": [
                summary_row["refined_iou_mean"],
                summary_row["refined_abs_offset_start_mean"],
                summary_row["refined_abs_offset_end_mean"],
                refined_abs_duration_error_mean,
            ],
        }
    )

    thesis_boundary_summary.to_csv(
        THESIS_SUMMARY_PATH,
        index=False,
    )

    print("Eventi matched inclusi:", int(summary_row["n_events"]))
    display(thesis_boundary_summary.round(3))
    print("Salvato:", THESIS_SUMMARY_PATH)

Tabella già disponibile: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\boundary_refinement\tables\pos_delay_boundary_refinement_thesis_summary.csv


,Metrica,Finestra raw,Finestra raffinata
0,IoU media,0.464,0.721
1,Errore assoluto inizio (giorni),5.622,1.764
2,Errore assoluto fine (giorni),2.317,1.239
3,Errore assoluto durata (giorni),7.538,2.406


### Interpretazione

- `event_results` contiene una riga per ogni detection matched, con i confini raw e raffinati.
- `overall_summary` sintetizza IoU e errori assoluti dei confini.
- `thesis_boundary_summary` è la tabella compatta per il confronto tra finestra raw e finestra raffinata.